# Spark Connect + Armada Demo

**Architecture:**
- Spark driver + Connect server run **in the cluster** (Armada cluster mode)
- This notebook is a **thin gRPC client** - no JVM locally

**Prerequisites:**
```bash
./scripts/runJupyter.sh -C
kubectl port-forward -n default $(kubectl get pod -n default -l spark-role=driver,spark-app-name=spark-connect-server -o name | head -1) 15002:15002
```

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Defaults assume the port-forward workflow: kubectl port-forward ... 15002:15002
# To use the Armada ingress instead, uncomment and edit the two lines below
# with the hostname from `kubectl get ingress` and the kind ingress hostPort.
HOST = os.environ.get("SPARK_CONNECT_HOST", "host.docker.internal")
PORT = os.environ.get("SPARK_CONNECT_PORT", "15002")
# HOST = "driver-15002-armada-<jobid>-0.default.svc"   # ingress hostname (also in /etc/hosts -> 127.0.0.1)
# PORT = "9999"                                         # kind ingress hostPort
CONNECT_URL = f"sc://{HOST}:{PORT}"

# If JWT auth is enabled on the Connect server (SPARK_ARMADA_CONNECT_OWNER set
# at submit time), pass SPARK_CONNECT_TOKEN to runJupyter.sh so it forwards
# into this container. The token is attached as the gRPC Authorization header.
TOKEN = os.environ.get("SPARK_CONNECT_TOKEN", "")
if TOKEN:
    CONNECT_URL = f"{CONNECT_URL}/;token={TOKEN}"

spark = SparkSession.builder.remote(CONNECT_URL).getOrCreate()
print(f"Connected! Spark version: {spark.version}")

In [ ]:
spark.sql("SELECT 'Hello from Spark Connect on Armada!' as message").show(truncate=False)

## Dynamic Allocation Test

200 partitions with 10s delay each to trigger executor scale-up.

Watch with: `kubectl get pods -n default -w`

In [ ]:
import time

num_partitions = 200

print(f"Running Spark Pi with {num_partitions} slow partitions...")

def slow_pi_partition(iterator):
    import pandas as pd
    import time
    import random
    for pdf in iterator:
        time.sleep(10)
        count = 0
        total = len(pdf)
        for _ in range(total):
            x, y = random.random(), random.random()
            if x * x + y * y < 1:
                count += 1
        yield pd.DataFrame({"count": [count], "total": [total]})

start = time.time()

df_input = spark.range(0, 1000000, numPartitions=num_partitions)
df_result = df_input.mapInPandas(slow_pi_partition, schema="count long, total long")
totals = df_result.agg(
    F.sum("count").alias("count"),
    F.sum("total").alias("total")
).collect()

pi = 4.0 * totals[0]["count"] / totals[0]["total"]
elapsed = time.time() - start

print(f"Pi is approximately: {pi}")
print(f"Completed in {elapsed:.1f}s")

In [ ]:
spark.stop()
print("Disconnected")